# L1b: Choosing and Building Data Representations

Primitive values rarely appear alone. In this lab, you will organize them using tuples, arrays, sets, and dictionaries, choosing each representation according to the operations the problem requires.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Choose a collection:__ Distinguish tuples, arrays, sets, and dictionaries by the operations each one supports, and select among them by asking which operations the problem requires rather than which container is most familiar.
> * __Predict valid operations:__ Explain why indexing, mutation, and key lookup succeed on some containers and raise errors on others, and say which of the three a given container will accept before you run the code.
> * __Interpret container types:__ Read the element, key, and value types encoded in Julia's collection types, and use them to predict whether a proposed update will succeed.

For each task, predict the result, run the cells, and use the Things to think about questions to explain what happened.

Let's get started!
___

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Besides Julia's `Base` library, this lab uses [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) for the checks at the end. The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), which this lab does not call.

___

## Task 1: Tuples as fixed records
Collection types group multiple values into a single container; examples include tuples, arrays, sets, and dictionaries, whose elements may themselves be primitive values or other collections. 

A [tuple](https://docs.julialang.org/en/v1/manual/functions/#Tuples) is an immutable, ordered collection with a fixed length that may contain values of different types, making it useful for grouping related values without a mutable container. This immutability is shallow: tuple slots cannot be reassigned, but a mutable object stored in a slot, such as an array, can still be changed in place.

Let's explore tuples with a concrete example. Since [Tuple types](https://docs.julialang.org/en/v1/base/base/#Core.Tuple) are immutable, they can't be changed once constructed.

The `example_tuple::Tuple{Int64, Float64}` variable holds two values of different types: an age in years and a measurement. The type itself records both the length and the element types.

In [2]:
example_tuple = let
    pair = (18,36.6); # populate with data. Notice not the same type for each element
end;

What is the type of the `example_tuple` variable? Let's use [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) to find out.

In [3]:
typeof(example_tuple)

Tuple{Int64, Float64}

Tuples are immutable. Let's try to change a value in the `example_tuple::Tuple{Int64, Float64}` variable. This should blow up, because [Tuples in Julia](https://docs.julialang.org/en/v1/base/base/#Core.Tuple) are immutable.

> **Try-catch blocks:** The `try-catch` construct allows us to handle errors gracefully instead of crashing the program. 
> Code in the `try` block is executed, and if an error occurs, execution jumps to the `catch` block where we can handle the error (like printing a message) rather than terminating the program.  The program continues executing normally after the `catch` block.

So what happens?

In [4]:
try
    example_tuple[1] = 6 # this will raise an error because tuples are immutable
catch e
    println("expected error: ", e)
end
println("After the try-catch block, the program continues executing normally.")

expected error: MethodError(setindex!, ((18, 36.6), 6, 1), 0x0000000000009885)
After the try-catch block, the program continues executing normally.


What does the bitstring look like for the `example_tuple::Tuple{Int64, Float64}` variable?

In [5]:
try
    bitstring(example_tuple) # Can't get the bitstring directly; a Tuple is not a primitive type.
catch e
    println("expected error: ", e)
end

expected error: ArgumentError("Tuple{Int64, Float64} not a primitive type")


However, we can get the elements of `example_tuple` and their bit layouts by [indexing into the Tuple](https://docs.julialang.org/en/v1/base/base/#Core.Tuple). For example, let's look at the second element:

In [6]:
bitstring(example_tuple[2]) # get the bitstring of the component i

"0100000001000010010011001100110011001100110011001100110011001101"

We can see the raw bytes associated with the `example_tuple::Tuple{Int64,Float64}` using [the `reinterpret(...)` function](https://docs.julialang.org/en/v1/base/arrays/#Base.reinterpret). Note: this works because the tuple is composed of `isbits` elements and the total size aligns; the exact byte order and layout you see will reflect host endianness and alignment.

In [7]:
v = reinterpret(NTuple{16,UInt8}, example_tuple) |> collect # we have 16 8-bit blocks (128 bits total)

16-element Vector{UInt8}:
 0x12
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0x00
 0xcd
 0xcc
 0xcc
 0xcc
 0xcc
 0x4c
 0x42
 0x40

### Things to think about
* __Why can a tuple be read but not changed?__ The assignment `example_tuple[1] = 6` failed, but reading `example_tuple[2]` succeeded. What does that tell you about the difference between reading and reassigning a tuple slot? What type would you expect Julia to infer for `("sensor-A", 18.2, true)`?

___

## Task 2: Arrays as mutable sequences
An array is a contiguous, ordered collection of elements of the same type, allowing constant-time access to its elements via integer indices. In most languages, arrays occupy a single block of memory, with element access computed as the base address plus the index times the memory size of each element.

> **Julia vs. Python arrays:** [Julia's `Array{T}` type](https://docs.julialang.org/en/v1/base/arrays/#Core.Array-Tuple%7BNothing,%20Any%7D) is a built-in, statically typed container that is `1-indexed` and stored [in column-major order](https://en.wikipedia.org/wiki/Row-_and_column-major_order). Python's native lists are heterogeneous and zero-indexed, while [NumPy's homogeneous arrays](https://numpy.org/doc/stable/reference/generated/numpy.array.html) are zero-indexed and row-major (implemented in a separate C library rather than the core language).

Arrays in both Julia and Python are mutable, meaning elements can be changed after we populate the array. Let's explore a Julia array:

In [8]:
a = rand(10) # build a 10-element random array

10-element Vector{Float64}:
 0.6543001138661924
 0.11640774627168449
 0.8036417336982323
 0.7183501927356095
 0.43816159898373186
 0.16999360982316336
 0.8558717914285026
 0.08269231930360033
 0.30890926409637975
 0.6373832567254217

We access the elements of an array by passing the index of the array in square brackets, e.g., `a[3]` returns the third element in Julia (because it is `1`-based):

In [9]:
a[3]

0.8036417336982323

Arrays are __mutable__, i.e., we can change them after we build them. For example:

In [10]:
a[3] = π

π = 3.1415926535897...

Arrays in Julia are `1`-based, unlike C, Python, and Java, which are `0`-based.
> __Note:__ This is a deliberate choice, and Julia is in good company: Fortran, MATLAB, and R (the languages scientific computing grew up on) are all `1`-based. The practical argument is that indices line up with the mathematics you are transcribing. When you write $\sum_{i=1}^{n}a_{i}$, the loop is `for i ∈ 1:n` and `a[1]` really is $a_{1}$. 
> 
> However, this can be a source of confusion for those coming from `0`-based languages. Most algorithms in the CS literature are written `0`-based, so translating them takes care.

What happens if we try to grab an element that is _outside_ the array?

In [11]:
try
    a[11] # asking for index 11, but the array has only 10 items
catch e
    println("expected error: ", e)
end

expected error: BoundsError([0.6543001138661924, 0.11640774627168449, 3.141592653589793, 0.7183501927356095, 0.43816159898373186, 0.16999360982316336, 0.8558717914285026, 0.08269231930360033, 0.30890926409637975, 0.6373832567254217], (11,))


### Things to think about
* __What does an array's element type permit?__ The assignment `a[3] = π` succeeded. Based on `a`'s type, what would happen if you assigned a value that cannot be converted to `Float64`, and why?

___

## Task 3: Sets and dictionaries for membership and lookup
A [Set type](https://docs.julialang.org/en/v1/base/collections/#Base.Set) is an unordered collection of unique elements that supports fast membership checks, insertions, and removals. A [Dictionary (or map) is an associative container](https://docs.julialang.org/en/v1/base/collections/#Base.Dict) that stores key–value pairs, allowing lookup, insertion, and deletion of values based on their unique keys.

> **Julia vs. Python collections:** Julia's `Set{T}` and `Dict{K,V}` are parametric containers, meaning every element in a `Set` has the same type `T`, and every key–value pair in a `Dict` has types `K` and `V`. However, the elements can be any type `T`, and the keys `K` and values `V` can also be of any type. 
> 
> On the other hand, Python's built-in `set` and `dict` are heterogeneous by default, because each slot holds a generic `object` reference. Julia can do the same with `Set{Any}` and `Dict{Any,Any}`; the difference is that Julia makes you ask for that flexibility.

Let's build a few examples of set and dictionary collection types. The `d::Dict{Int64, String}` variable models the lines of a text file: each key is a line number, and each value is the text on that line.

In [12]:
d = let

    d = Dict{Int64, String}(); # creates a dictionary that models text in a file.
    d[1] = "This is the first line in a text file";
    d[2] = "This is the second line in a text file";
    d[3] = "This is the last line in a text file";

    d
end

Dict{Int64, String} with 3 entries:
  2 => "This is the second line in a text file"
  3 => "This is the last line in a text file"
  1 => "This is the first line in a text file"

We can access the values stored in a dictionary by passing in the `key` pointing to a `value`. Indexing with a key the dictionary does not hold raises an error, so when you are unsure, ask first with [the `haskey(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.haskey). Line `2` is there, so we can index for it directly:

In [13]:
d[2]

"This is the second line in a text file"

A dictionary is accessed by key, not by position; a set is queried by membership, not by position. The order in which either collection is displayed is not part of its representation, so do not write code that depends on that order.

Consider the `s::Set{Char}` example:

In [14]:
s = let

    s = Set{Char}(); # empty at this point
    push!(s, 'a'); # add items to the set using `push!`
    push!(s, 'b');
    push!(s, 'c');
    push!(s, 'd');

    s
end

Set{Char} with 4 elements:
  'a'
  'c'
  'd'
  'b'

We can't access a particular item in the `s::Set{Char}` set by passing in an index (or key) because these concepts don't apply to sets. Instead, we can use [the `pop!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pop!) to pop (get) an arbitrary element from a set:

> __Popping removes as well as returns.__ Calling `pop!(s)` mutates `s`, so calling it a second time takes a second element out. If you want to start over, re-run the cell that builds `s`.

Let's take an element out and see which one we get:

In [15]:
pop!(s)

'a': ASCII/Unicode U+0061 (category Ll: Letter, lowercase)

All the typical mathematical operations on sets, such as intersection, union, or membership checks, are implemented in most modern programming languages, including Julia; [see the documentation for operations on sets in Julia](https://docs.julialang.org/en/v1/base/collections/#Set-Like-Collections).

### Things to think about
* __Membership or lookup?__ A set answers whether a value is present, whereas a dictionary retrieves a value using a key. Which would you choose for storing unique chemical compound names, and which would you choose for looking up molecular weights by chemical compound name? Why?

___

## Lab check

Run this cell after completing the notebook. These checks verify the representations and mutations demonstrated above; the Things to think about questions focus on the reasoning the tests cannot capture.


In [16]:
let
    @testset verbose = true "CHEME 4/5800 L1b Representation Check" begin
        @test example_tuple isa Tuple{Int64,Float64}
        @test a isa Vector{Float64}
        @test a[3] == Float64(π)
        @test d isa Dict{Int64,String}
        @test d[2] == "This is the second line in a text file"
        @test s isa Set{Char}
        @test length(s) == 3
    end
end;

Test Summary:                         | Pass  Total  Time
CHEME 4/5800 L1b Representation Check |    7      7  0.6s


___


## Summary

Choosing a representation means choosing which operations the program should support.

> __Key Takeaways:__
>
> * **A container is a set of operations:** Tuples fix their contents at construction, arrays give ordered indexed access you can overwrite, sets answer membership without order, and dictionaries answer lookup by key, so naming the operation you need also names the container you need.
> * **Mutability belongs to the representation:** Tuple slots cannot be reassigned, while arrays, sets, and dictionaries can be updated after construction, so whether the data must change helps determine the container.
> * **Element types constrain valid updates:** An array such as `Vector{Float64}` accepts values that can be converted to `Float64`, while typed sets and dictionaries similarly enforce their declared element, key, and value types.

Week 2 builds on these representations by defining interfaces and performing transformations, grouping, and summarization.
___